In [1]:
import os
os.environ.get('TRAJECTORY_WORKDIR'), os.environ.get('PROJECT_TAG')

('/workspace/015f36e1-0a1c-4aed-a9a3-1d1924983c4a', None)

In [2]:
import pandas as pd, numpy as np, os
WD = "/workspace/015f36e1-0a1c-4aed-a9a3-1d1924983c4a"
RAW = f"{WD}/data/drop-tests/accelerometer-tuning/raw"
os.listdir(RAW)[:5], os.listdir(RAW)

(['06.02.2026_Signal9.csv',
  '06.02.2026_Signal1.csv',
  '06.02.2026_Signal6.csv',
  '06.02.2026_Signal8.csv',
  '06.02.2026_Signal3.csv'],
 ['06.02.2026_Signal9.csv',
  '06.02.2026_Signal1.csv',
  '06.02.2026_Signal6.csv',
  '06.02.2026_Signal8.csv',
  '06.02.2026_Signal3.csv',
  '06.02.2026_Signal7.csv',
  '06.02.2026.csv',
  '06.02.2026_Signal12.csv',
  '06.02.2026_Signal4.csv',
  '06.02.2026_Signal13.csv',
  '06.02.2026_Signal11.csv',
  '06.02.2026_Signal5.csv',
  '06.02.2026_Signal10.csv',
  '06.02.2026_Signal2.csv'])

In [3]:
with open(f"{RAW}/06.02.2026_Signal1.csv") as f:
    for _ in range(10):
        print(f.readline().rstrip())
print("---SERIES---")
with open(f"{RAW}/06.02.2026.csv") as f:
    for _ in range(20):
        print(f.readline().rstrip())

TP4 Time Domain Export File
Generated From: C:\TP4_46\DataStore\EventDatabase.TP4_db
Recording Session ID: n0jdwk
Recording Session Date: 06/02/2026 03:07:29 PM,
Event: 1
EventTime: 6/2/2026 9:39:23 PM
Order: 0

Time (sec), CH1 Acc (G's), CH2 Acc (G's), CH3 Acc (G's), CH4 Acc (G's),
0.000000E+000, -8.896233E+000, 1.451244E+000, 4.067771E+000, 4.363969E+000
---SERIES---
TP4 Series Table Export File
Generated From: C:\TP4_46\DataStore\EventDatabase.TP4_db
Recording Session ID: n0jdwk
Recording Session Date: 06/02/2026 03:07:29 PM,
TriggerMethod,EventNumber,EventTime,Order,Accel CH1: Channel 1, Duration, Delta V, Accel CH2: Channel 2, Duration, Delta V, Accel CH3: Channel 3, Duration, Delta V, Accel CH4: Channel 4, Duration, Delta V,
 , , , ,G, msec , in/sec, G, msec , in/sec, G, msec , in/sec, G, msec , in/sec,
0,1,06/02/2026 03:39:23 PM,0,2576.40, 0.40, 204.70, 209.89, 0.06, 2.66, 161.39, 0.07, 2.15, 1279.87, 0.61, 152.23,
0,2,06/02/2026 03:40:36 PM,0,8803.49, 0.66, 1362.41, 208.67, 0.0

In [4]:
def load_event(n):
    path = f"{RAW}/06.02.2026_Signal{n}.csv"
    # Find header row
    with open(path) as f:
        lines = f.readlines()
    for i, line in enumerate(lines):
        if line.startswith("Time"):
            header_row = i
            break
    df = pd.read_csv(path, skiprows=header_row, skipinitialspace=True)
    df.columns = [c.strip().rstrip(',').strip() for c in df.columns]
    # drop trailing empty
    df = df.loc[:, ~df.columns.str.match(r'Unnamed')]
    return df

ev1 = load_event(1)
print(ev1.shape)
print(ev1.columns.tolist())
ev1.head(3)

(25000, 5)
['Time (sec)', "CH1 Acc (G's)", "CH2 Acc (G's)", "CH3 Acc (G's)", "CH4 Acc (G's)"]


,Time (sec),CH1 Acc (G's),CH2 Acc (G's),CH3 Acc (G's),CH4 Acc (G's)
0,0.000000,-8.896233,1.451244,4.067771,4.363969
1,0.000008,-8.880473,1.743220,4.028452,4.217800
2,0.000016,-8.905238,2.023103,4.366241,4.284388


In [5]:
# Load all events
events = {n: load_event(n) for n in range(1, 14)}
fs = 1/(events[1]["Time (sec)"].iloc[1] - events[1]["Time (sec)"].iloc[0])
print("fs =", fs)
# Sanity check sampling interval consistency
for n, df in events.items():
    dt = np.diff(df["Time (sec)"].values)
    print(n, df.shape, "dt min/max:", dt.min(), dt.max(), "channels max abs:",
          [round(df[c].abs().max(),2) for c in df.columns[1:]])

fs = 125000.0
1 (25000, 5) dt min/max: 7.999999999980245e-06 8.000000000008e-06 channels max abs: [np.float64(2576.4), np.float64(209.89), np.float64(161.39), np.float64(1279.87)]
2 (25000, 5) dt min/max: 7.999999999980245e-06 8.000000000008e-06 channels max abs: [np.float64(8803.49), np.float64(208.67), np.float64(128.9), np.float64(829.81)]
3 (25000, 5) dt min/max: 7.999999999980245e-06 8.000000000008e-06 channels max abs: [np.float64(8806.21), np.float64(58.3), np.float64(34.11), np.float64(231.57)]
4 (25000, 5) dt min/max: 7.999999999980245e-06 8.000000000008e-06 channels max abs: [np.float64(2492.92), np.float64(213.91), np.float64(184.61), np.float64(1301.05)]
5 (25000, 5) dt min/max: 7.999999999980245e-06 8.000000000008e-06 channels max abs: [np.float64(8806.1), np.float64(71.13), np.float64(45.27), np.float64(215.26)]
6 (25000, 5) dt min/max: 7.999999999980245e-06 8.000000000008e-06 channels max abs: [np.float64(20.28), np.float64(0.58), np.float64(0.44), np.float64(0.31)]
7 (2

In [6]:
# (a) Saturation/clipping check
# Look at CH1 in events 2,3,5 (and others for comparison). Examine the max plateau and quantization granularity.
import numpy as np

def check_clip(df, ch, near=0.999):
    s = df[ch].values
    mx = s.max(); mn = s.min()
    # Count samples within near*max
    n_near_top = int((s >= near*mx).sum())
    # Find longest plateau
    near_mask = s >= near*mx
    if near_mask.any():
        # longest run
        runs = np.diff(np.where(np.concatenate(([near_mask[0]],
                                                np.diff(near_mask.astype(int))!=0,
                                                [True])))[0])
        # Just take longest run of True
        longest = 0; cur = 0
        for v in near_mask:
            if v: cur += 1; longest = max(longest, cur)
            else: cur = 0
    else:
        longest = 0
    # Quantization step (unique values near top)
    top_vals = np.unique(s[s >= 0.95*mx])
    return mx, mn, n_near_top, longest, top_vals[-5:] if len(top_vals)>=5 else top_vals

for n in [1,2,3,4,5,9,10,13]:
    df = events[n]
    mx, mn, ntop, longest, topv = check_clip(df, "CH1 Acc (G's)")
    print(f"Event {n}: CH1 max={mx:.3f} min={mn:.3f} samples>=99.9%max={ntop} longest_plateau={longest} samples ({longest*8} µs) top vals: {topv}")

Event 1: CH1 max=2576.396 min=-1646.308 samples>=99.9%max=1 longest_plateau=1 samples (8 µs) top vals: [2456.484 2473.282 2536.703 2557.689 2576.396]
Event 2: CH1 max=8803.494 min=-2473.679 samples>=99.9%max=8 longest_plateau=8 samples (64 µs) top vals: [8800.788 8801.581 8802.769 8802.886 8803.494]
Event 3: CH1 max=8806.206 min=-2587.399 samples>=99.9%max=7 longest_plateau=7 samples (56 µs) top vals: [8802.513 8804.217 8805.07  8805.241 8806.206]
Event 4: CH1 max=2492.923 min=-1731.887 samples>=99.9%max=1 longest_plateau=1 samples (8 µs) top vals: [2378.457 2391.693 2421.426 2485.807 2492.923]
Event 5: CH1 max=8806.097 min=-2605.654 samples>=99.9%max=7 longest_plateau=7 samples (56 µs) top vals: [8803.195 8804.613 8805.717 8805.746 8806.097]
Event 9: CH1 max=932.229 min=-337.202 samples>=99.9%max=7 longest_plateau=7 samples (56 µs) top vals: [931.9901 931.9972 932.2006 932.2082 932.2286]
Event 10: CH1 max=932.124 min=-325.487 samples>=99.9%max=6 longest_plateau=6 samples (48 µs) top v

In [7]:
# Important observation: Events 9, 10, 13 all peak at ~932 G with multi-sample plateaus too.
# Look more carefully — check the precise top value to see if events 9/10/13 are also clipping at a ~932 G ceiling.
# Also look at the negative rail.

def top_plateau_detail(df, ch):
    s = df[ch].values
    # top 10 unique values
    top10 = np.sort(np.unique(s))[-10:]
    # negative rail
    bot10 = np.sort(np.unique(s))[:10]
    # find runs where s is within 0.5 G of max
    near = s >= (s.max() - 0.5)
    idx = np.where(near)[0]
    return top10, bot10, idx

for n in [1,2,3,4,5,9,10,13]:
    top10, bot10, idx = top_plateau_detail(events[n], "CH1 Acc (G's)")
    print(f"\nEvent {n}: top-10 unique = {top10}")
    print(f"   bottom-10 unique = {bot10}")
    print(f"   indices where value within 0.5G of max: count={len(idx)} indices(first 20)={idx[:20]}")



Event 1: top-10 unique = [2268.485 2270.784 2320.211 2366.302 2415.121 2456.484 2473.282 2536.703
 2557.689 2576.396]
   bottom-10 unique = [-1646.308 -1634.341 -1624.493 -1588.384 -1573.507 -1524.842 -1488.923
 -1459.312 -1399.783 -1372.99 ]
   indices where value within 0.5G of max: count=1 indices(first 20)=[550]

Event 2: top-10 unique = [8788.826 8793.406 8795.42  8798.31  8798.895 8800.788 8801.581 8802.769
 8802.886 8803.494]
   bottom-10 unique = [-2473.679 -2463.355 -2409.607 -2197.215 -2069.954 -1996.695 -1903.481
 -1898.97  -1879.755 -1857.393]
   indices where value within 0.5G of max: count=1 indices(first 20)=[467]

Event 3: top-10 unique = [8792.033 8795.115 8797.226 8799.851 8802.017 8802.513 8804.217 8805.07
 8805.241 8806.206]
   bottom-10 unique = [-2587.399 -2539.121 -2529.859 -2253.132 -2200.573 -2108.674 -2031.793
 -2030.096 -2019.247 -2013.064]
   indices where value within 0.5G of max: count=1 indices(first 20)=[469]

Event 4: top-10 unique = [2266.706 2300.172

In [8]:
# Strong evidence: events 9, 10, 13 all peak at ~932 G with a multi-sample plateau at samples 523-528 (~4.2 ms)
# AND the top values are not at the typical sat ceiling but close.
# Critical observation: events 9, 10, 13 also peak at *exactly* time index ~525 (~4.2 ms),
# which is the trigger artifact time — and they max around 932 G which is suspiciously identical.
# Let's check if 932 G is also a clip / artifact, or is a real impact peak.

# Also, events 2, 3, 5 peak at index ~467-471, NOT at the trigger time of ~525.
# So events 2/3/5 saturation is at a different time than the trigger artifact, good.
# Let's identify the time of CH1 max in each event:
for n in range(1,14):
    df = events[n]
    s = df["CH1 Acc (G's)"].values
    t = df["Time (sec)"].values
    imx = int(np.argmax(s))
    print(f"Event {n}: CH1 max={s[imx]:.2f} at t={t[imx]*1000:.3f} ms (idx={imx})")

Event 1: CH1 max=2576.40 at t=4.400 ms (idx=550)
Event 2: CH1 max=8803.49 at t=3.736 ms (idx=467)
Event 3: CH1 max=8806.21 at t=3.752 ms (idx=469)
Event 4: CH1 max=2492.92 at t=4.392 ms (idx=549)
Event 5: CH1 max=8806.10 at t=3.760 ms (idx=470)
Event 6: CH1 max=20.28 at t=85.832 ms (idx=10729)
Event 7: CH1 max=21.70 at t=4.512 ms (idx=564)
Event 8: CH1 max=1.59 at t=81.192 ms (idx=10149)
Event 9: CH1 max=932.23 at t=4.200 ms (idx=525)
Event 10: CH1 max=932.12 at t=4.208 ms (idx=526)
Event 11: CH1 max=2.74 at t=13.904 ms (idx=1738)
Event 12: CH1 max=0.99 at t=11.520 ms (idx=1440)
Event 13: CH1 max=933.77 at t=4.208 ms (idx=526)


In [9]:
# CH4 peak times:
for n in range(1,14):
    df = events[n]
    s4 = df["CH4 Acc (G's)"].values
    t = df["Time (sec)"].values
    imx = int(np.argmax(np.abs(s4)))
    print(f"Event {n}: CH4 max|s|={s4[imx]:.2f} at t={t[imx]*1000:.3f} ms (idx={imx})  CH1 raw peak time")


Event 1: CH4 max|s|=1279.87 at t=4.192 ms (idx=524)  CH1 raw peak time
Event 2: CH4 max|s|=829.81 at t=31.112 ms (idx=3889)  CH1 raw peak time
Event 3: CH4 max|s|=231.57 at t=3.944 ms (idx=493)  CH1 raw peak time
Event 4: CH4 max|s|=1301.05 at t=4.184 ms (idx=523)  CH1 raw peak time
Event 5: CH4 max|s|=215.26 at t=3.952 ms (idx=494)  CH1 raw peak time
Event 6: CH4 max|s|=0.31 at t=145.096 ms (idx=18137)  CH1 raw peak time
Event 7: CH4 max|s|=0.49 at t=5.480 ms (idx=685)  CH1 raw peak time
Event 8: CH4 max|s|=0.42 at t=0.952 ms (idx=119)  CH1 raw peak time
Event 9: CH4 max|s|=15.56 at t=4.384 ms (idx=548)  CH1 raw peak time
Event 10: CH4 max|s|=-3.82 at t=148.584 ms (idx=18573)  CH1 raw peak time
Event 11: CH4 max|s|=-158.83 at t=3.896 ms (idx=487)  CH1 raw peak time
Event 12: CH4 max|s|=-176.49 at t=3.896 ms (idx=487)  CH1 raw peak time
Event 13: CH4 max|s|=157.94 at t=4.384 ms (idx=548)  CH1 raw peak time


In [10]:
# Now (b) channel-to-sensor mapping check.
# Single-axis sensor should be independent; tri-axis CH2,CH3,CH4 should share spectral content / common-mode trigger spikes etc.
# Let's compute cross-correlation / co-occurrence of high-frequency excitation on impact events (1,4,11,12).

from scipy import signal

def psd(df, ch):
    s = df[ch].values - df[ch].values[:200].mean()
    f, P = signal.welch(s, fs=125000, nperseg=4096)
    return f, P

# Event 11 — CH1 silent, CH2-4 active
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for n, ax in zip([1, 4, 11, 12], axes.flat):
    df = events[n]
    for ch in ["CH1 Acc (G's)", "CH2 Acc (G's)", "CH3 Acc (G's)", "CH4 Acc (G's)"]:
        f, P = psd(df, ch)
        ax.loglog(f, P, label=ch.split()[0])
    ax.set_title(f"Event {n}")
    ax.legend(fontsize=8)
    ax.set_xlabel("Hz"); ax.set_ylabel("PSD (G²/Hz)")
plt.tight_layout()
plt.savefig(f"{WD}/psd_check.png", dpi=120)
plt.close()
print("ok")

ok


In [11]:
# Check inter-channel correlation around the impact for events 1,4 to confirm CH2/3/4 are a group:
for n in [1, 4, 11, 12]:
    df = events[n]
    # Take a 5 ms window centered around the impact (e.g., t=4 ms)
    t = df["Time (sec)"].values
    mask = (t >= 0.002) & (t <= 0.010)
    sub = df.loc[mask, ["CH1 Acc (G's)","CH2 Acc (G's)","CH3 Acc (G's)","CH4 Acc (G's)"]]
    corr = sub.corr()
    print(f"\nEvent {n} corr (2-10ms):\n{corr.round(2)}")


Event 1 corr (2-10ms):
               CH1 Acc (G's)  CH2 Acc (G's)  CH3 Acc (G's)  CH4 Acc (G's)
CH1 Acc (G's)           1.00          -0.00           0.05           0.12
CH2 Acc (G's)          -0.00           1.00          -0.05           0.33
CH3 Acc (G's)           0.05          -0.05           1.00          -0.29
CH4 Acc (G's)           0.12           0.33          -0.29           1.00

Event 4 corr (2-10ms):
               CH1 Acc (G's)  CH2 Acc (G's)  CH3 Acc (G's)  CH4 Acc (G's)
CH1 Acc (G's)           1.00           0.02           0.06           0.12
CH2 Acc (G's)           0.02           1.00          -0.14           0.43
CH3 Acc (G's)           0.06          -0.14           1.00          -0.22
CH4 Acc (G's)           0.12           0.43          -0.22           1.00

Event 11 corr (2-10ms):
               CH1 Acc (G's)  CH2 Acc (G's)  CH3 Acc (G's)  CH4 Acc (G's)
CH1 Acc (G's)           1.00           0.89           0.91           0.91
CH2 Acc (G's)           0.89           

In [12]:
# Interesting: in events 11/12, ALL channels are highly correlated, meaning whatever pulse hit was sensed by everything.
# In events 1/4, CH1 is uncorrelated with CH2-4 (because CH1 sits at a different location/sensor)
# CH2/CH3/CH4 in events 1/4 are not super tightly correlated either, but CH2/CH4 share 0.33-0.43.
# Pre-impact / quiescent noise tells us more about whether 2,3,4 belong to one sensor.

for n in [1, 4]:
    df = events[n]
    t = df["Time (sec)"].values
    mask = t < 0.002  # pre-trigger noise
    sub = df.loc[mask, ["CH1 Acc (G's)","CH2 Acc (G's)","CH3 Acc (G's)","CH4 Acc (G's)"]]
    print(f"\nEvent {n} pre-trigger (0-2ms) std:\n{sub.std().round(3)}")
    print(f"corr:\n{sub.corr().round(2)}")


Event 1 pre-trigger (0-2ms) std:
CH1 Acc (G's)    1.171
CH2 Acc (G's)    0.623
CH3 Acc (G's)    0.256
CH4 Acc (G's)    0.226
dtype: float64
corr:
               CH1 Acc (G's)  CH2 Acc (G's)  CH3 Acc (G's)  CH4 Acc (G's)
CH1 Acc (G's)           1.00          -0.07           0.02          -0.07
CH2 Acc (G's)          -0.07           1.00           0.35           0.01
CH3 Acc (G's)           0.02           0.35           1.00           0.01
CH4 Acc (G's)          -0.07           0.01           0.01           1.00

Event 4 pre-trigger (0-2ms) std:
CH1 Acc (G's)    0.807
CH2 Acc (G's)    0.292
CH3 Acc (G's)    0.205
CH4 Acc (G's)    0.227
dtype: float64
corr:
               CH1 Acc (G's)  CH2 Acc (G's)  CH3 Acc (G's)  CH4 Acc (G's)
CH1 Acc (G's)           1.00          -0.02           0.09          -0.19
CH2 Acc (G's)          -0.02           1.00           0.33          -0.03
CH3 Acc (G's)           0.09           0.33           1.00          -0.15
CH4 Acc (G's)          -0.19          -0

In [13]:
# Noise floor: CH1 is the noisiest, CH2-4 quieter. CH2/CH3 share ~0.33 correlation; CH4 is more independent in noise.
# Hmm — that's a bit unusual. If CH2/3/4 were one tri-axis ICP, you'd expect similar noise floors,
# possibly with some correlated common-mode (from cable / DAQ ground), but each axis has independent piezo crystal noise.
# CH2 σ=0.62 / CH3 σ=0.26 / CH4 σ=0.23 — CH2 is 2.5x noisier than CH3/4. Mild concern.

# Critical test of the README's "CH2/CH3/CH4 move together as a group" claim:
# In events 11/12 ALL FOUR move together (corr ~0.9+). That suggests a common-mode pulse, NOT proof that 2/3/4 are one sensor.
# Look at events where the tri-axis (whichever it is) has biggest amplitude on Z (impact axis).
# Events with high CH4: 1, 2, 4 — that's the "impact axis"
# Events 11/12 have CH4 as the biggest of CH2/3/4 (-158, -176) — consistent with Z impact axis assignment.

# Time-of-peak alignment as a tri-axis check (all 3 axes should respond at the same instant):
for n in [1, 4, 11, 12]:
    df = events[n]
    t = df["Time (sec)"].values
    for ch in ["CH2 Acc (G's)","CH3 Acc (G's)","CH4 Acc (G's)"]:
        s = df[ch].values
        i = int(np.argmax(np.abs(s)))
        print(f"Event {n} {ch.split()[0]}: peak |{s[i]:.2f}| at {t[i]*1000:.3f} ms")
    print()

Event 1 CH2: peak |209.89| at 6.056 ms
Event 1 CH3: peak |161.39| at 6.648 ms
Event 1 CH4: peak |1279.87| at 4.192 ms

Event 4 CH2: peak |-213.91| at 6.672 ms
Event 4 CH3: peak |184.61| at 35.624 ms
Event 4 CH4: peak |1301.05| at 4.184 ms

Event 11 CH2: peak |-108.56| at 3.896 ms
Event 11 CH3: peak |-91.21| at 3.896 ms
Event 11 CH4: peak |-158.83| at 3.896 ms

Event 12 CH2: peak |-124.55| at 3.896 ms
Event 12 CH3: peak |-115.91| at 3.896 ms
Event 12 CH4: peak |-176.49| at 3.896 ms



In [14]:
# Excellent: events 11/12 ALL three of CH2,CH3,CH4 peak at exactly 3.896 ms — strong proof they're three axes of one sensor.
# Event 1/4: CH4 peaks early but CH2/3 ringing peaks later — consistent with strong Z impact + off-axis ringing.
# So channel mapping CH2-4 = tri-axis is consistent. CH4 = Z (impact-aligned) — confirmed (largest of three on impact events).
# However event 1/4 CH3 peaks at 6.6-35 ms — those are ringing peaks, not the impact transient.

# Now (c) Validate the CFC filter. SAE J211 specifies:
# - Anti-alias: sample rate >= 8 * CFC class (CFC1000 needs >= 8 kHz sampling = OK; we have 125 kHz)
# - Filter: 4-pole phaseless Butterworth, forward+backward (so effective 8-pole), -3 dB at 1.65*CFC
# - CFC180 -3dB ~ 300 Hz, CFC1000 -3dB ~ 1.65 kHz
# - 1/T (impulse duration) constraints: T (filter delay) ~ 2 ms for CFC60, 0.3 ms for CFC1000
# The script already does phaseless butter — let me check & confirm.
import scipy.signal as ss

def cfc_filter(x, fs, cfc):
    """SAE J211 CFC filter: 4-pole Butterworth lowpass, applied forward-backward (zero-phase).
    Cutoff = 1.65*CFC Hz."""
    fc = 1.65 * cfc
    b, a = ss.butter(4, fc/(fs/2), 'low')
    return ss.filtfilt(b, a, x)

# Quick verification: compare our filter output to peak_summary
ps = pd.read_csv(f"{WD}/data/drop-tests/accelerometer-tuning/peak_summary.csv")
print(ps.columns.tolist())
ps.head()

['event', 'CH1_raw', 'CH2_raw', 'CH3_raw', 'CH4_raw', 'CH1_cfc1000', 'CH2_cfc1000', 'CH3_cfc1000', 'CH4_cfc1000', 'CH1_cfc180', 'CH2_cfc180', 'CH3_cfc180', 'CH4_cfc180', 'tri_resultant_cfc180', 'CH1_over_CH4_cfc180', 'CH1_saturated']


,event,CH1_raw,CH2_raw,CH3_raw,CH4_raw,CH1_cfc1000,CH2_cfc1000,CH3_cfc1000,CH4_cfc1000,CH1_cfc180,CH2_cfc180,CH3_cfc180,CH4_cfc180,tri_resultant_cfc180,CH1_over_CH4_cfc180,CH1_saturated
0,1,2576.396,209.891,161.394,1279.866,1761.406,78.314,-52.661,1060.039,482.862,16.826,7.836,311.628,312.136,1.549,0
1,2,8803.494,208.666,128.901,829.807,8493.076,-75.165,22.637,107.159,2722.986,36.441,11.229,38.483,38.594,70.758,1
2,3,8806.206,-58.300,34.115,231.575,8475.075,-21.625,28.375,157.147,2657.639,-9.554,14.255,39.895,40.091,66.616,1
3,4,2492.923,-213.908,184.608,1301.054,1725.494,106.547,-32.375,1075.330,472.878,26.951,9.075,315.957,317.086,1.497,0
4,5,8806.097,71.130,45.271,215.256,8464.560,-17.160,26.009,153.617,2663.933,-5.725,10.230,39.465,39.523,67.501,1


In [15]:
# Verify by reproducing CFC180 peaks on event 1:
df = events[1]
for ch in ["CH1 Acc (G's)","CH2 Acc (G's)","CH3 Acc (G's)","CH4 Acc (G's)"]:
    s = df[ch].values
    s_dc = s - s[:200].mean()
    f180 = cfc_filter(s_dc, 125000, 180)
    f1000 = cfc_filter(s_dc, 125000, 1000)
    print(f"{ch}: CFC180 max={f180.max():.2f} min={f180.min():.2f}; CFC1000 max={f1000.max():.2f}")
# Compare with the script output for event 1:
print("Script: CH1 CFC180=482.86, CH4=311.63 ✓")

CH1 Acc (G's): CFC180 max=487.19 min=-209.00; CFC1000 max=1566.14
CH2 Acc (G's): CFC180 max=10.38 min=-7.14; CFC1000 max=68.05
CH3 Acc (G's): CFC180 max=3.22 min=-8.86; CFC1000 max=6.30
CH4 Acc (G's): CFC180 max=229.08 min=-39.42; CFC1000 max=972.71
Script: CH1 CFC180=482.86, CH4=311.63 ✓


In [16]:
# Slight differences likely because the script doesn't subtract DC, or uses a different butter design
# Let me check the script for exact implementation:
import pathlib
print(pathlib.Path(f"{WD}/scripts/analysis/accelerometer_tuning_analysis.py").read_text())

#!/usr/bin/env python3
"""Analysis of the drop-tower accelerometer "tuning" data (issue #71).

Me (@me-madsen) and @ctrhjk ran a series of drop-tower tests on 06/02/2026 to
standardize the test setup and to understand why the **single-axis** and
**tri-axis** accelerometers do not report the same acceleration. The raw data
are TP4 (Test Partner 4) exports:

* ``06.02.2026.csv``        -- series *table* export: one row per event with the
  per-channel peak ``Accel``, pulse ``Duration`` and ``Delta V``.
* ``06.02.2026_SignalN.csv`` -- *time-domain* export for event ``N`` (N = 1..13),
  4 channels sampled at 125 kHz for 0.2 s (25 000 samples).

Channel mapping (inferred from the data; see report -- pending confirmation of
the per-test position labels):

* ``CH1``           -> single-axis accelerometer (impact direction only)
* ``CH2, CH3, CH4`` -> tri-axis accelerometer (X, Y, Z); CH4 is its impact axis

The script:

1. Parses the table and time-domain exports.
2. Plots every event (all fo

In [17]:
print(pathlib.Path(f"{WD}/scripts/analysis/accelerometer_tuning_analysis.py").read_text()[3000:7000])

-------------------------------------------------- #
def cfc_filter(x: np.ndarray, fs: float, cfc: float) -> np.ndarray:
    """Apply the SAE J211 Channel Frequency Class filter.

    Implements the 2-pole Butterworth low-pass from SAE J211-1 Appendix C run
    forward then backward to give a phaseless 4-pole response. ``cfc`` is the
    Channel Frequency Class (e.g. 180, 1000); the -3 dB point is ~1.66 * cfc.
    """
    T = 1.0 / fs
    wd = 2.0 * math.pi * cfc * 2.0775
    wa = math.tan(wd * T / 2.0)
    den = 1.0 + math.sqrt(2.0) * wa + wa * wa
    a0 = wa * wa / den
    a1 = 2.0 * a0
    a2 = a0
    b1 = -2.0 * (wa * wa - 1.0) / den
    b2 = (-1.0 + math.sqrt(2.0) * wa - wa * wa) / den

    def _pass(sig: np.ndarray) -> np.ndarray:
        y = np.empty_like(sig)
        y[0] = sig[0]
        y[1] = sig[1]
        for i in range(2, len(sig)):
            y[i] = (
                a0 * sig[i] + a1 * sig[i - 1] + a2 * sig[i - 2]
                + b1 * y[i - 1] + b2 * y[i - 2]
        

In [18]:
# OK — the script uses the *official* SAE J211 difference-equation form (Appendix C of J211/1).
# That's the correct implementation. My scipy 4-pole butterworth filtfilt was slightly different
# (because filtfilt of a 4-pole filter gives effective 8th-order; SAE J211 specifies a 2-pole forward+backward = 4th-order net).
# The script's form is canonical SAE J211 (4-pole effective, not 8-pole). Good.

# Cutoff used: 1.65 * cfc in -3dB definition. Let me check what the script actually uses.
# Script: wd = 2*pi*cfc*2.0775 — this is the digital cutoff transform per SAE J211.
# Per SAE J211/1 (2014):
#  - Fc filter freq = CFC * 2.0775 used in the bilinear transform; -3dB ≈ CFC * 1.65
# So script uses canonical SAE J211 Appendix C ✓

# Now (d) confirm/extend the ~4.2 ms trigger/magnet-release artifact.
# Look at CH4 peak times across events that lack real impact vs those with impact:

# Build a richer picture: time of peak |CH4| in a small window (3.5–5 ms) for every event
for n in range(1, 14):
    df = events[n]
    t = df["Time (sec)"].values
    s = df["CH4 Acc (G's)"].values
    mask = (t >= 0.003) & (t <= 0.005)
    if mask.any():
        sub = s[mask]; tsub = t[mask]
        i = int(np.argmax(np.abs(sub)))
        print(f"Event {n}: CH4 within 3-5 ms — peak {sub[i]:.2f} G at t={tsub[i]*1000:.3f} ms")

Event 1: CH4 within 3-5 ms — peak 1279.87 G at t=4.192 ms
Event 2: CH4 within 3-5 ms — peak -184.71 G at t=4.696 ms
Event 3: CH4 within 3-5 ms — peak 231.57 G at t=3.944 ms
Event 4: CH4 within 3-5 ms — peak 1301.05 G at t=4.184 ms
Event 5: CH4 within 3-5 ms — peak 215.26 G at t=3.952 ms
Event 6: CH4 within 3-5 ms — peak 0.27 G at t=4.208 ms
Event 7: CH4 within 3-5 ms — peak 0.40 G at t=4.952 ms
Event 8: CH4 within 3-5 ms — peak 0.38 G at t=3.856 ms
Event 9: CH4 within 3-5 ms — peak 15.56 G at t=4.384 ms
Event 10: CH4 within 3-5 ms — peak 1.57 G at t=4.392 ms
Event 11: CH4 within 3-5 ms — peak -158.83 G at t=3.896 ms
Event 12: CH4 within 3-5 ms — peak -176.49 G at t=3.896 ms
Event 13: CH4 within 3-5 ms — peak 157.94 G at t=4.384 ms


In [19]:
# Critical observation:
# - In "quiet" events 6,7,8 the CH4 peak in 3-5 ms is only ~0.3 G — NOT a fixed trigger artifact on CH4.
# - The README claims "CH4 carries a fixed ~4.2 ms trigger/magnet-release artifact"; that's NOT what the data show.
# - Events 6,7,8 (aborted drops) have no consistent transient at 4.2 ms on CH4.
# - The ~4.2 ms timing on impact events is the actual *impact* of the carriage with the floor (the system trigger fires near the
#   release; impact occurs ~4 ms later after the drop fall). So the recurring 4.2 ms peak IS the impact, not an artifact.
# Let's confirm by looking at the CH4 pulse shape in event 1.

# Plot event 1 CH4 around 3-6 ms
fig, ax = plt.subplots(figsize=(10, 4))
df = events[1]
t = df["Time (sec)"].values * 1000  # ms
for ch, color in zip(["CH1 Acc (G's)","CH2 Acc (G's)","CH3 Acc (G's)","CH4 Acc (G's)"], ['k','C0','C1','C2']):
    s = df[ch].values
    ax.plot(t, s, label=ch.split()[0], lw=0.6, color=color)
ax.set_xlim(0, 10); ax.set_xlabel("ms"); ax.set_ylabel("G"); ax.legend(); ax.set_title("Event 1: zoomed")
plt.tight_layout()
plt.savefig(f"{WD}/ev1_zoom.png", dpi=120); plt.close()
print("Event 1 saved")

# Pulse duration on CH4 around 4.2 ms — characteristic of impact (~0.5 ms FWHM is typical for hard impacts)
# vs a "trigger artifact" which is usually a single-sample electrical spike.
s4 = events[1]["CH4 Acc (G's)"].values
# Find the FWHM around the peak
i_pk = np.argmax(s4)
half = s4[i_pk] / 2
# scan outward
i_lo = i_pk
while i_lo > 0 and s4[i_lo] > half: i_lo -= 1
i_hi = i_pk
while i_hi < len(s4)-1 and s4[i_hi] > half: i_hi += 1
print(f"Event 1 CH4 peak idx={i_pk}, FWHM ~ {(i_hi-i_lo)*8} µs")

Event 1 saved
Event 1 CH4 peak idx=524, FWHM ~ 280 µs


In [20]:
# 280 µs FWHM — that's a real mechanical impact pulse, not a 1-sample spike.
# Now check CH1 pulse FWHM (claimed by README to differ in shape):
s1 = events[1]["CH1 Acc (G's)"].values
i_pk = np.argmax(s1)
half = s1[i_pk] / 2
i_lo = i_pk
while i_lo > 0 and s1[i_lo] > half: i_lo -= 1
i_hi = i_pk
while i_hi < len(s1)-1 and s1[i_hi] > half: i_hi += 1
print(f"Event 1 CH1 peak idx={i_pk}, t={i_pk*8} µs, FWHM ~ {(i_hi-i_lo)*8} µs")
# Time between CH1 peak (550) and CH4 peak (524) is (550-524)*8 = 208 µs
print("ΔT(CH1 peak - CH4 peak):", (550-524)*8, "µs")

Event 1 CH1 peak idx=550, t=4400 µs, FWHM ~ 184 µs
ΔT(CH1 peak - CH4 peak): 208 µs


In [21]:
# CH1 raw peak is 208 µs (=26 samples) LATER than CH4 peak. That's a big offset, which could be:
# - the two sensors are at different mechanical locations (different shock travel times)
# - or CH1 is showing ringing not impact (its raw peak is mount resonance riding on the rigid-body pulse)
# Let's check after CFC-180 (rigid-body) — peaks of both should be aligned (both responding to rigid-body deceleration).

s1_180 = cfc_filter(s1 - s1[:200].mean(), 125000, 180)
s4_180 = cfc_filter(s4 - s4[:200].mean(), 125000, 180)
# Override with the J211 implementation:
import math
def j211(x, fs, cfc):
    T = 1.0/fs
    wd = 2.0*math.pi*cfc*2.0775
    wa = math.tan(wd*T/2.0)
    den = 1.0 + math.sqrt(2.0)*wa + wa*wa
    a0 = wa*wa/den; a1 = 2.0*a0; a2 = a0
    b1 = -2.0*(wa*wa-1.0)/den
    b2 = (-1.0 + math.sqrt(2.0)*wa - wa*wa)/den
    def _pass(sig):
        y = np.empty_like(sig); y[0] = sig[0]; y[1] = sig[1]
        for i in range(2, len(sig)):
            y[i] = a0*sig[i] + a1*sig[i-1] + a2*sig[i-2] + b1*y[i-1] + b2*y[i-2]
        return y
    fwd = _pass(x); bwd = _pass(fwd[::-1])[::-1]
    return bwd

s1_180 = j211(s1.copy(), 125000, 180)
s4_180 = j211(s4.copy(), 125000, 180)
s1_1000 = j211(s1.copy(), 125000, 1000)
s4_1000 = j211(s4.copy(), 125000, 1000)
print("CH1 CFC180 peak:", s1_180.max(), "at idx", np.argmax(s1_180), "t=", np.argmax(s1_180)*8, "µs")
print("CH4 CFC180 peak:", s4_180.max(), "at idx", np.argmax(s4_180), "t=", np.argmax(s4_180)*8, "µs")
print("CH1 CFC1000 peak:", s1_1000.max(), "at idx", np.argmax(s1_1000), "t=", np.argmax(s1_1000)*8, "µs")
print("CH4 CFC1000 peak:", s4_1000.max(), "at idx", np.argmax(s4_1000), "t=", np.argmax(s4_1000)*8, "µs")

CH1 CFC180 peak: 482.8620749055371 at idx 1978 t= 15824 µs
CH4 CFC180 peak: 311.6281161463418 at idx 522 t= 4176 µs
CH1 CFC1000 peak: 1761.4057675728968 at idx 555 t= 4440 µs
CH4 CFC1000 peak: 1060.0394405454913 at idx 524 t= 4192 µs


In [22]:
# Very important: CH1 CFC180 peak occurs at 15.8 ms — not the impact at 4.2 ms!
# This means CH1 has a low-frequency post-impact response (maybe long-period mount oscillation or DC offset settling)
# that exceeds the impact pulse in the CFC180 filter.
# This is a *huge* finding the original analysis missed.
# Let's plot CH1 CFC180 to see what's happening:

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
df = events[1]
t = df["Time (sec)"].values * 1000
s1 = df["CH1 Acc (G's)"].values
s4 = df["CH4 Acc (G's)"].values
s1_180 = j211(s1.copy(), 125000, 180)
s4_180 = j211(s4.copy(), 125000, 180)
s1_1000 = j211(s1.copy(), 125000, 1000)
s4_1000 = j211(s4.copy(), 125000, 1000)
ax[0].plot(t, s1, 'k-', lw=0.4, label='CH1 raw', alpha=0.5)
ax[0].plot(t, s1_1000, 'b-', lw=0.7, label='CH1 CFC1000')
ax[0].plot(t, s1_180, 'r-', lw=1.0, label='CH1 CFC180')
ax[0].legend(); ax[0].set_ylabel("CH1 (G)"); ax[0].set_title("Event 1 — CH1 CFC180 peak is at 15.8 ms, NOT impact (4.2 ms)")
ax[1].plot(t, s4, 'k-', lw=0.4, label='CH4 raw', alpha=0.5)
ax[1].plot(t, s4_1000, 'b-', lw=0.7, label='CH4 CFC1000')
ax[1].plot(t, s4_180, 'r-', lw=1.0, label='CH4 CFC180')
ax[1].legend(); ax[1].set_ylabel("CH4 (G)"); ax[1].set_xlabel("ms"); ax[1].set_xlim(0, 50)
plt.tight_layout()
plt.savefig(f"{WD}/ev1_cfc.png", dpi=130); plt.close()
print("plot saved")


plot saved


In [23]:
# Look at CH1 raw and CFC180 between 4 ms and 50 ms — is there a low-frequency post-impact oscillation?
# Print values
df = events[1]
s1 = df["CH1 Acc (G's)"].values
s1_180 = j211(s1.copy(), 125000, 180)
# DC offset of CH1
print("CH1 first 200 samples mean:", s1[:200].mean(), "std:", s1[:200].std())
print("CH1 first 200 CFC180:", s1_180[:200].mean())
# value at 15.8 ms peak
print("CH1 CFC180 at 15.8 ms peak:", s1_180[1978])
print("CH1 CFC180 at 4.2 ms (impact):", s1_180[525])
# values across time
import numpy as np
for tt in [0, 2, 4, 4.4, 6, 8, 10, 12, 14, 16, 18, 20, 30, 40, 50]:
    idx = int(tt*125)
    if idx < len(s1_180):
        print(f"  t={tt:5.1f} ms: raw={s1[idx]:8.2f}  CFC180={s1_180[idx]:8.2f}")

CH1 first 200 samples mean: -9.25191821 std: 0.895568788680041
CH1 first 200 CFC180: -9.278574469675693
CH1 CFC180 at 15.8 ms peak: 482.8620749055371
CH1 CFC180 at 4.2 ms (impact): 302.05163312339965
  t=  0.0 ms: raw=   -8.90  CFC180=   -8.62
  t=  2.0 ms: raw=   -7.91  CFC180=  -18.17
  t=  4.0 ms: raw=    6.44  CFC180=  237.83
  t=  4.4 ms: raw= 2576.40  CFC180=  333.18
  t=  6.0 ms: raw=  -88.27  CFC180=  209.16
  t=  8.0 ms: raw= -297.98  CFC180=   39.83
  t= 10.0 ms: raw=  232.36  CFC180=  242.79
  t= 12.0 ms: raw=  238.04  CFC180=  379.18
  t= 14.0 ms: raw=  396.52  CFC180=  460.70
  t= 16.0 ms: raw=  581.99  CFC180=  481.19
  t= 18.0 ms: raw=  342.47  CFC180=  440.26
  t= 20.0 ms: raw=  437.93  CFC180=  385.84
  t= 30.0 ms: raw=  -24.82  CFC180=  -28.98
  t= 40.0 ms: raw= -204.39  CFC180= -132.48
  t= 50.0 ms: raw= -167.92  CFC180= -155.78


In [24]:
# Ah! The CH1 raw is actually 396, 581, 342 G at 14-18 ms — there is a real low-frequency response on CH1
# centered at ~16 ms (likely a structural/mount oscillation after impact, perhaps the carriage rebound or sled body mode).
# So CH1's CFC180 "peak" of 482 G is not the primary impact pulse — it's a secondary response.
# This means the CH1/CH4 = 1.5x ratio (483/312) is comparing CH1 secondary response to CH4 primary impact —
# NOT a valid sensitivity ratio.

# This is a substantial correction to the original analysis. The right comparison would be:
# CFC1000 (peak g) — both peak near impact (4.2-4.4 ms):
#   CH1 CFC1000 = 1761 G (idx 555, t=4.44 ms) — but wait, even CFC1000 may be contaminated by ringing
# Let's check the CH1 raw spectrum: PSD shows energy out to 20 kHz, so CFC1000 (1.65 kHz cutoff) should suppress it.

# Let me also check CFC1000 peak time across events
for n in [1, 4]:
    df = events[n]
    s1 = df["CH1 Acc (G's)"].values; s4 = df["CH4 Acc (G's)"].values
    s1_1000 = j211(s1.copy(), 125000, 1000); s4_1000 = j211(s4.copy(), 125000, 1000)
    s1_180 = j211(s1.copy(), 125000, 180); s4_180 = j211(s4.copy(), 125000, 180)
    i1k1 = np.argmax(s1_1000); i4k1 = np.argmax(s4_1000)
    i1k180 = np.argmax(s1_180); i4k180 = np.argmax(s4_180)
    print(f"\nEvent {n}:")
    print(f"  CH1 CFC1000 peak = {s1_1000[i1k1]:.1f} G at t={i1k1*8/1000:.3f} ms")
    print(f"  CH4 CFC1000 peak = {s4_1000[i4k1]:.1f} G at t={i4k1*8/1000:.3f} ms")
    print(f"  ratio at CH4 impact time (CH1[i4]/CH4[i4]): {s1_1000[i4k1]/s4_1000[i4k1]:.2f}")
    print(f"  CH1 CFC180 peak = {s1_180[i1k180]:.1f} G at t={i1k180*8/1000:.3f} ms")
    print(f"  CH4 CFC180 peak = {s4_180[i4k180]:.1f} G at t={i4k180*8/1000:.3f} ms")
    print(f"  CH1 CFC180 at CH4 impact time = {s1_180[i4k180]:.1f}")



Event 1:
  CH1 CFC1000 peak = 1761.4 G at t=4.440 ms
  CH4 CFC1000 peak = 1060.0 G at t=4.192 ms
  ratio at CH4 impact time (CH1[i4]/CH4[i4]): 0.23
  CH1 CFC180 peak = 482.9 G at t=15.824 ms
  CH4 CFC180 peak = 311.6 G at t=4.176 ms
  CH1 CFC180 at CH4 impact time = 295.5



Event 4:
  CH1 CFC1000 peak = 1725.5 G at t=4.432 ms
  CH4 CFC1000 peak = 1075.3 G at t=4.192 ms
  ratio at CH4 impact time (CH1[i4]/CH4[i4]): 0.23
  CH1 CFC180 peak = 472.9 G at t=15.744 ms
  CH4 CFC180 peak = 316.0 G at t=4.176 ms
  CH1 CFC180 at CH4 impact time = 299.6


In [25]:
# MASSIVELY important finding:
# - At the actual impact time (4.18-4.19 ms), CH1 CFC1000 is much LOWER than CH4 CFC1000:
#   CH1[i_impact] = 0.23 * CH4[i_impact] (events 1 & 4 both)
# - BUT CH1's CFC1000 peak (1761 G) occurs LATER (4.44 ms), 248 µs after CH4's impact peak.
# This time offset suggests CH1 is mounted at a *different location* (mechanically downstream of the impact)
# OR CH1 has slower response / different sensor location, picking up a different acceleration regime.

# This contradicts the original "single-axis is 1.5x tri-axis on CFC180" finding.
# The 1.5x came from comparing CH1's LATE LF post-impact response (15.8 ms) to CH4's primary impact (4.18 ms).
# Those are different events on the structure, not the same input!

# So events 1 and 4 are NOT a valid cross-calibration either, contrary to what the report claims.
# CH1 and CH4 are at different locations on different structural members, seeing different motions.

# Also: events 9, 10, 13 all peak at exactly 4.20 ms (idx 525-526) with a multi-sample plateau at ~932 G.
# Let's check if 932 G is a CH1 saturation/range limit
print("Events 9/10/13 CH1 max idx and plateau:")
for n in [9, 10, 13]:
    df = events[n]
    s1 = df["CH1 Acc (G's)"].values
    # Find plateau
    mx = s1.max()
    near = s1 >= mx - 1.0
    idx_near = np.where(near)[0]
    print(f"  Event {n}: max={mx:.2f}, plateau samples within 1G: {idx_near}, count={len(idx_near)}")
    # Surrounding values
    if len(idx_near) > 0:
        i = idx_near[0]
        print(f"  surrounding (idx {i-3}..{i+10}): {s1[i-3:i+11]}")

Events 9/10/13 CH1 max idx and plateau:
  Event 9: max=932.23, plateau samples within 1G: [522 523 524 525 526 527 528], count=7
  surrounding (idx 519..532): [929.5848 930.5132 931.0688 931.5362 931.9901 932.2006 932.2286 932.2082
 931.9972 931.5587 931.0947 930.5959 929.7302 928.5193]
  Event 10: max=932.12, plateau samples within 1G: [522 523 524 525 526 527 528 529], count=8
  surrounding (idx 519..532): [928.8663 929.9564 930.7102 931.165  931.6269 932.0038 932.1102 932.1237
 932.08   931.7388 931.1857 930.6913 930.0575 929.0374]
  Event 13: max=933.77, plateau samples within 1G: [522 523 524 525 526 527 528], count=7
  surrounding (idx 519..532): [931.0488 932.0122 932.6272 933.0279 933.4216 933.6857 933.7631 933.7665
 933.6411 933.2672 932.73   932.2159 931.5151 930.3781]


In [26]:
# That's NOT a clipping plateau — it's a smooth rounded peak (Gaussian-ish, ~7 samples wide at <1G of max).
# A clip would show a flat line at exactly one value (or one DAC code). These show smooth ~1 G/sample curvature
# even within the "near max" set — these are real peaks, not clips.

# In contrast, look at events 2/3/5:
print("Events 2/3/5 CH1 surrounding peak:")
for n in [2, 3, 5]:
    df = events[n]
    s1 = df["CH1 Acc (G's)"].values
    mx = s1.max()
    near = s1 >= mx - 5.0
    idx_near = np.where(near)[0]
    print(f"  Event {n}: max={mx:.2f}, plateau samples within 5G: count={len(idx_near)}")
    i = idx_near[0]
    j = idx_near[-1]
    print(f"  values within 5G: {s1[i:j+1]}")
    print(f"  step-to-step deltas: {np.diff(s1[i:j+1])}")
    print()

Events 2/3/5 CH1 surrounding peak:
  Event 2: max=8803.49, plateau samples within 5G: count=6
  values within 5G: [8801.581 8802.769 8803.494 8802.886 8800.788 8798.895]
  step-to-step deltas: [ 1.188  0.725 -0.608 -2.098 -1.893]

  Event 3: max=8806.21, plateau samples within 5G: count=6
  values within 5G: [8802.017 8804.217 8805.241 8806.206 8805.07  8802.513]
  step-to-step deltas: [ 2.2    1.024  0.965 -1.136 -2.557]

  Event 5: max=8806.10, plateau samples within 5G: count=5
  values within 5G: [8804.613 8805.717 8806.097 8805.746 8803.195]
  step-to-step deltas: [ 1.104  0.38  -0.351 -2.551]



In [27]:
# Hmm — this is interesting. The peaks at ~8806 G in events 2, 3, 5 are NOT a flat clip plateau either.
# They show <3 G/sample curvature near peak (consistent with a heavily filtered or band-limited "real" peak being approached).
# Let me check more carefully: the README claimed a "flat plateau for ~0.2 ms" — that's 25 samples at 125 kHz.
# We see only 5-6 samples within 5 G of peak. That's NOT 0.2 ms plateau.

# BUT — three different events reaching essentially the SAME peak (8803, 8806, 8806 G) is still suspicious.
# Could it be the +5V analog full-scale of an ICP/IEPE conditioner divided by sensitivity?
# E.g., a +-5V ICP with 0.5 mV/G sensitivity → 10000 G full scale.
#       a +-5V ICP with 0.567 mV/G → 8800 G full scale.
# But the lack of a flat plateau says the ADC is NOT clipping at the same DAC code.
# It might be: the sensor itself reached its mechanical limit ('saturated' in the sensor's pre-amp),
# producing a smoothed ceiling, OR these 3 drops were genuinely from the same height producing the same peak.

# Look at the timing of the 8806 peaks: events 2/3/5 all peak at t = 3.74/3.75/3.76 ms.
# And the LATE LF peak on CH1 in events 1/4 (the 482 G CFC180 secondary response) occurs at ~15.8 ms.
# Where do events 2/3/5 CH1 CFC180 peaks come from?
for n in [2, 3, 5]:
    df = events[n]
    s1 = df["CH1 Acc (G's)"].values
    s1_180 = j211(s1.copy(), 125000, 180)
    s1_1000 = j211(s1.copy(), 125000, 1000)
    i180 = np.argmax(s1_180); i1k = np.argmax(s1_1000)
    print(f"Event {n}: CH1 raw peak={s1.max():.1f} at idx={np.argmax(s1)} (t={np.argmax(s1)*8/1000:.2f}ms);"
          f" CFC1000 peak={s1_1000[i1k]:.1f} at t={i1k*8/1000:.2f}ms;"
          f" CFC180 peak={s1_180[i180]:.1f} at t={i180*8/1000:.2f}ms")

Event 2: CH1 raw peak=8803.5 at idx=467 (t=3.74ms); CFC1000 peak=8493.1 at t=3.74ms; CFC180 peak=2723.0 at t=3.71ms
Event 3: CH1 raw peak=8806.2 at idx=469 (t=3.75ms); CFC1000 peak=8475.1 at t=3.74ms; CFC180 peak=2657.6 at t=3.72ms


Event 5: CH1 raw peak=8806.1 at idx=470 (t=3.76ms); CFC1000 peak=8464.6 at t=3.75ms; CFC180 peak=2663.9 at t=3.72ms


In [28]:
# Events 2,3,5 CH1: raw 8803 G, CFC1000 8475 G, CFC180 2660-2720 G — all at t≈3.74 ms (Earlier than the typical impact at 4.2 ms!)
# These events have CH1 peaking 400 µs *earlier* than the typical impact. AND the CH4 in events 2/3/5 does NOT show a big impact pulse.
# Events 2,3,5 stand out: CH1 sees a giant 8800 G peak at 3.74 ms; tri-axis sees almost nothing (CH4 max 200-800 G, much smaller)
# AND the timing is different than events 1/4 impact.
# Strong suggestion: in events 2/3/5 the single-axis sensor (CH1) was at a *different location* than in events 1/4,
# possibly attached to something that experienced a larger and earlier shock (e.g., directly on the platen vs on the test article).

# So the "ceiling" interpretation of events 2/3/5 is suspect:
# - It's not a flat clipping plateau (only ~5 samples wide, with curvature visible)
# - The fact that 3 events reach the SAME ceiling (8803, 8806, 8806) is suspicious BUT could just mean
#   the platen impact happened reproducibly at the same height in those 3 tests.
# - Alternative explanation: CH1 sensor was bolted directly to the platen (or in a position seeing a different shock)
#   for runs 2/3/5, and the apparent saturation is actually the real sensor response.

# Let me check if 8806 G is right at the sensor's full scale. PCB 350-series single-axis ICPs come in 5kG, 10kG, 20kG, 100kG ranges.
# 8806 G ≈ 88% of 10000 G — borderline saturation for a 10kG sensor (90% of FSO is typical clip limit).
# A PCB 350B04 (5000g) would clip at ~5000-5500 G. A 350M77 (10000g) clips at ~10000-11000 G. A 350B23 (20000g) at ~20000 G.

# Let me check the values more precisely — is there a quantization signature near saturation?
# The 16-bit ADC quantization at +-10V FS with 0.567 mV/G sensitivity gives ~0.34 G/LSB.
# Real-vs-clipped test: check the histograms of differences
s1_ev2 = events[2]["CH1 Acc (G's)"].values
# Compute first difference distribution near peak
peak_idx = np.argmax(s1_ev2)
near = s1_ev2[peak_idx-20:peak_idx+20]
print("CH1 event 2 near peak:")
print(near)
print("First diff:", np.diff(near))

CH1 event 2 near peak:
[5151.539 5626.877 6121.982 6661.231 7236.598 7820.275 8316.803 8598.162
 8677.855 8693.414 8719.238 8744.344 8759.573 8771.358 8781.687 8788.57
 8793.406 8798.31  8801.581 8802.769 8803.494 8802.886 8800.788 8798.895
 8795.42  8788.826 8780.563 8769.738 8759.962 8757.977 8751.322 8717.063
 8694.538 8710.488 8507.434 7815.57  6994.551 6527.014 6037.753 4998.379]
First diff: [ 4.753380e+02  4.951050e+02  5.392490e+02  5.753670e+02  5.836770e+02
  4.965280e+02  2.813590e+02  7.969300e+01  1.555900e+01  2.582400e+01
  2.510600e+01  1.522900e+01  1.178500e+01  1.032900e+01  6.883000e+00
  4.836000e+00  4.904000e+00  3.271000e+00  1.188000e+00  7.250000e-01
 -6.080000e-01 -2.098000e+00 -1.893000e+00 -3.475000e+00 -6.594000e+00
 -8.263000e+00 -1.082500e+01 -9.776000e+00 -1.985000e+00 -6.655000e+00
 -3.425900e+01 -2.252500e+01  1.595000e+01 -2.030540e+02 -6.918640e+02
 -8.210190e+02 -4.675370e+02 -4.892610e+02 -1.039374e+03]


In [29]:
# Look at this rise: 475 G/sample, 495, 539, 575, 583, 496, 281, 79, 15 ... and then it sits within ±5 G for ~12 samples,
# then drops 22, 16, then jumps -203, -692, -821, -467 ...
# THIS IS THE TEXTBOOK SIGNATURE OF A SENSOR SATURATING:
# - the rising slew shows decreasing slope as the sensor approaches its limit (475 → 583 → 496 → 281 → 79 → 15 G/sample)
# - then the signal sits within a narrow ±5 G band for ~12 samples (compressed peak)
# - then on the falling edge, normal slew resumes (-203, -692, -821 G/sample — comparable to the rising slope magnitude)
# The asymmetry between fast rise/fall and slow plateau IS sensor saturation/compression.

# The original "0.2 ms flat plateau" claim is mostly right in spirit (compressed top), but it's NOT a digital ADC clip
# (which would be perfectly flat at one DAC code). It's an analog sensor saturation/compression — the sensor's
# physical or amplifier limit being reached.

# Let me find the plateau width: count samples between "rising fast" and "falling fast"
peak_idx = np.argmax(s1_ev2)
# rising-edge end: last sample where slope > 50 G/sample before peak
slopes_pre = np.diff(s1_ev2[peak_idx-30:peak_idx])
slopes_post = np.diff(s1_ev2[peak_idx:peak_idx+30])
# A simple measure: count samples within (max - 100 G) of peak
near_band = s1_ev2 >= s1_ev2[peak_idx] - 100
# find longest run containing the peak
i0 = peak_idx
while i0 > 0 and near_band[i0]: i0 -= 1
i1 = peak_idx
while i1 < len(near_band)-1 and near_band[i1]: i1 += 1
print(f"Event 2 CH1 'flat top' within 100 G of peak: {i1-i0} samples = {(i1-i0)*8} µs")
# So about a 100 µs compressed peak

for n in [2, 3, 5]:
    s1 = events[n]["CH1 Acc (G's)"].values
    peak_idx = np.argmax(s1)
    near_band = s1 >= s1[peak_idx] - 100
    i0 = peak_idx
    while i0 > 0 and near_band[i0]: i0 -= 1
    i1 = peak_idx
    while i1 < len(near_band)-1 and near_band[i1]: i1 += 1
    print(f"Event {n} CH1 compressed peak (within 100G of max): {i1-i0} samples = {(i1-i0)*8} µs")

Event 2 CH1 'flat top' within 100 G of peak: 23 samples = 184 µs
Event 2 CH1 compressed peak (within 100G of max): 23 samples = 184 µs
Event 3 CH1 compressed peak (within 100G of max): 22 samples = 176 µs
Event 5 CH1 compressed peak (within 100G of max): 24 samples = 192 µs


In [30]:
# Excellent — there IS a ~180-190 µs compressed top (close to the README's "~0.2 ms").
# The ceiling is real, but the mechanism is analog sensor/conditioner saturation (smooth compression),
# not a digital ADC clip (flat at one code).
# So the original interpretation of "saturating at 8806 G" is essentially correct;
# just refine the *mechanism* from "DAQ clip" to "sensor full-scale saturation/compression".

# Now (e) sensitivity mismatch check.
# Key fact: in events 9, 10, 13 CH1 peaks at exactly 932, 932, 933 G at exactly t=4.20 ms — very reproducible.
# This is unlikely to be a sensor full-scale because we have higher values (2576, 8806). So events 9-10-13 reflect a different,
# smaller drop with very reproducible peak — or they're some sort of artifact.
# Tri-axis (CH2-4) for these events is essentially noise. So sensor positioning was different.

# Hmm... 932/8806 = 0.106 — wait, an interesting check: 932 ≈ 8806/9.45. Not an obvious sensitivity ratio.
# Look at 8806 ~ G full scale. Looking at events 1/4 with both sensors reading: CH1 raw 2576, 2493 G.
# The single-axis CH1 raw values often look much larger than CH4 raw — but that's mostly mount ringing on CH1.

# Let me build the corrected peak summary: align peaks at impact (CH4 t=4.18-4.20 ms window):
def impact_window_peaks(df, ch4_impact_time_ms=4.18, search_ms=1.0):
    """Find each channel's peak within ±search_ms of CH4 impact time."""
    fs = 125000
    t = df["Time (sec)"].values * 1000  # ms
    s4 = df["CH4 Acc (G's)"].values
    # First, find CH4 impact time as the max |CH4| in 3-6 ms
    mask_impact = (t >= 3.0) & (t <= 6.0)
    i_impact = np.where(mask_impact)[0][np.argmax(np.abs(s4[mask_impact]))]
    t_impact = t[i_impact]
    results = {"CH4_impact_time_ms": t_impact, "impact_idx": i_impact}
    # Now define an analysis window of ±1 ms around impact
    mask = (t >= t_impact - search_ms) & (t <= t_impact + search_ms)
    for ch in ["CH1 Acc (G's)","CH2 Acc (G's)","CH3 Acc (G's)","CH4 Acc (G's)"]:
        s = df[ch].values
        s_dc = s - s[:200].mean()
        for cfc in (1000, 180):
            sf = j211(s_dc.copy(), fs, cfc)
            i = np.where(mask)[0][np.argmax(np.abs(sf[mask]))]
            results[f"{ch.split()[0]}_cfc{cfc}_peakG"] = float(sf[i])
            results[f"{ch.split()[0]}_cfc{cfc}_peakT_ms"] = float(t[i])
        # raw
        i_raw = np.where(mask)[0][np.argmax(np.abs(s_dc[mask]))]
        results[f"{ch.split()[0]}_raw_peakG"] = float(s_dc[i_raw])
        results[f"{ch.split()[0]}_raw_peakT_ms"] = float(t[i_raw])
    return results

ridge_rows = []
for n in range(1, 14):
    r = impact_window_peaks(events[n])
    r["event"] = n
    ridge_rows.append(r)
ridge_df = pd.DataFrame(ridge_rows)
ridge_df.set_index("event", inplace=True)
print(ridge_df[["CH4_impact_time_ms",
                "CH1_raw_peakG","CH1_cfc1000_peakG","CH1_cfc180_peakG",
                "CH4_raw_peakG","CH4_cfc1000_peakG","CH4_cfc180_peakG"]].round(1))

       CH4_impact_time_ms  CH1_raw_peakG  CH1_cfc1000_peakG  CH1_cfc180_peakG  \
event                                                                           
1                     4.2         2585.6             1770.7             342.6   
2                     4.7         8714.8             8404.3            2634.2   
3                     3.9         8783.0             8451.8            2634.4   
4                     4.2         2496.1             1728.6             339.6   
5                     4.0         8699.4             8357.8            2557.2   
6                     4.2            0.6                0.6               0.6   
7                     5.5           22.5                1.2               0.5   
8                     3.9           -9.2               -1.5              -0.4   
9                     4.4          920.7              881.4             258.8   
10                    4.4          918.7              879.8             256.9   
11                    3.9   

In [31]:
# Build the ratio analysis (events 1, 4) — proper at-impact-window peaks
# At impact, CH1/CH4 ratio:
for n in [1, 4]:
    r = ridge_df.loc[n]
    print(f"\nEvent {n} (in-window peaks):")
    print(f"  CH1 CFC1000 / CH4 CFC1000 = {r['CH1_cfc1000_peakG']/r['CH4_cfc1000_peakG']:.2f}")
    print(f"  CH1 CFC180  / CH4 CFC180  = {r['CH1_cfc180_peakG']/r['CH4_cfc180_peakG']:.2f}")
    print(f"  CH1 raw     / CH4 raw     = {r['CH1_raw_peakG']/r['CH4_raw_peakG']:.2f}")
    print(f"  Times of those peaks: CH1 CFC1000 t={r['CH1_cfc1000_peakT_ms']:.3f} ms, CH4 t={r['CH4_cfc1000_peakT_ms']:.3f} ms")
    print(f"                       CH1 CFC180 t={r['CH1_cfc180_peakT_ms']:.3f} ms, CH4 t={r['CH4_cfc180_peakT_ms']:.3f} ms")


Event 1 (in-window peaks):
  CH1 CFC1000 / CH4 CFC1000 = 1.68
  CH1 CFC180  / CH4 CFC180  = 1.12
  CH1 raw     / CH4 raw     = 2.03
  Times of those peaks: CH1 CFC1000 t=4.440 ms, CH4 t=4.192 ms
                       CH1 CFC180 t=4.416 ms, CH4 t=4.176 ms

Event 4 (in-window peaks):
  CH1 CFC1000 / CH4 CFC1000 = 1.62
  CH1 CFC180  / CH4 CFC180  = 1.10
  CH1 raw     / CH4 raw     = 1.93
  Times of those peaks: CH1 CFC1000 t=4.432 ms, CH4 t=4.192 ms
                       CH1 CFC180 t=4.408 ms, CH4 t=4.176 ms


In [32]:
# Critical correction to the original analysis:
# With CFC180 restricted to the impact window (not 200 ms of post-impact ringing), the CH1/CH4 ratio is ~1.10–1.12 NOT 1.55.
# The original 1.55 ratio was contaminated by including a LF post-impact response on CH1 that peaks 11 ms later (around 15.8 ms).
# When you compare the *actual impact peak*, the two sensors agree within ~10% on CFC180.
# On CFC1000, however, CH1 is 1.6–1.7x CH4. This larger ratio probably reflects:
#   1) different mounting locations (CH1 250 µs later than CH4 — different sensor position experiences slightly different shock)
#   2) or some residual mount resonance leaking past CFC1000 (1.65 kHz cutoff)

# Let me confirm by looking at the spectrum of CH1 below 1.65 kHz vs CH4:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for n, ax in zip([1, 4], axes):
    df = events[n]
    for ch, color in [("CH1 Acc (G's)",'r'), ("CH4 Acc (G's)",'b')]:
        s = df[ch].values - df[ch].values[:200].mean()
        f, P = signal.welch(s, fs=125000, nperseg=2048)
        ax.semilogy(f, P, label=ch.split()[0], color=color)
    ax.axvline(1650, color='k', ls='--', alpha=0.5, label='CFC1000 -3dB')
    ax.axvline(300, color='k', ls=':', alpha=0.5, label='CFC180 -3dB')
    ax.set_xlim(0, 5000); ax.set_xlabel("Hz"); ax.set_title(f"Event {n} PSD")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{WD}/psd_low.png", dpi=120); plt.close()
print("saved")

saved


In [33]:
# Check Nyquist / anti-alias adequacy. 125 kHz sampling. For CFC1000 (-3dB 1.65 kHz, stopband ~3-4 kHz),
# Nyquist is 62.5 kHz — far beyond needed. SAE J211 minimum sampling rate is 8 * CFC = 8 kHz for CFC1000.
# 125 kHz is 125x the required rate. Excellent.
# However the analog anti-alias filter of the DAQ must be < Nyquist (62.5 kHz). For CFC180 (300 Hz cutoff),
# we'd ideally have a digital anti-alias > 1 kHz but well below 62.5 kHz. With 125 kHz raw, we can see content out to
# 62.5 kHz. The PSDs show energy out to ~30 kHz, suggesting either: (a) high natural frequency response of sensor/mount,
# or (b) inadequate analog anti-aliasing.

# Now let me dump full picture by computing in-window CH4 (Z) vs tri-axis resultant
for n in [1, 4]:
    df = events[n]
    t = df["Time (sec)"].values * 1000
    # CFC180 filtered
    s2 = j211(df["CH2 Acc (G's)"].values - df["CH2 Acc (G's)"].values[:200].mean(), 125000, 180)
    s3 = j211(df["CH3 Acc (G's)"].values - df["CH3 Acc (G's)"].values[:200].mean(), 125000, 180)
    s4 = j211(df["CH4 Acc (G's)"].values - df["CH4 Acc (G's)"].values[:200].mean(), 125000, 180)
    R = np.sqrt(s2**2 + s3**2 + s4**2)
    mask = (t >= 3.5) & (t <= 5.5)
    print(f"Event {n}: tri-axis CFC180 resultant peak (in 3.5-5.5 ms) = {R[mask].max():.1f} G at t={t[mask][R[mask].argmax()]:.3f} ms")

Event 1: tri-axis CFC180 resultant peak (in 3.5-5.5 ms) = 307.8 G at t=4.176 ms
Event 4: tri-axis CFC180 resultant peak (in 3.5-5.5 ms) = 310.4 G at t=4.176 ms


In [34]:
# Tri-axis resultant ~308-310 G (at impact), vs CH1 CFC180 at impact window = 339-342 G.
# CH1/resultant = 1.10 — almost exactly the same as CH1/CH4 in-window ratio.
# The 1.55 ratio reported by the original analysis is an artifact of using the global CFC180 peak (post-impact LF response on CH1).

# Summary so far:
# (a) Saturation: Real. The 8800 G ceiling on CH1 events 2/3/5 is sensor saturation/compression
#     (not a flat digital clip), reached for ~180 µs. Real peak is unknown (greater than 8806 G).
#     Number of events: 3 (events 2, 3, 5). Identical ceilings (8803/8806/8806) ≈ within 3 G strongly suggests a fixed limit.
# (b) Channel mapping: CH2/3/4 = one tri-axis sensor is supported (events 11/12 show all three peaking at the same 3.896 ms,
#     with CH4 always the largest = Z impact axis). CH1 = single-axis. Confirmed.
# (c) CFC filtering: SAE J211 4-pole phaseless implementation is correct. 125 kHz sampling >> 8*CFC required. Adequate.
#     But the original analysis took the GLOBAL CFC180 peak in the 200-ms window, which conflates the impact pulse with later
#     structural responses. CFC180 peaks should be evaluated in the impact window (e.g., ±2 ms of CH4 impact).
# (d) "4.2 ms trigger artifact" claim is WRONG. Quiet events 6/7/8 do NOT show a peak at 4.2 ms on CH4.
#     The 4.2 ms peak is the actual impact (~280 µs FWHM pulse with ringing). Events 11/12 show all 4 channels peaking simultaneously
#     at 3.896 ms but that's just a smaller shock the tri-axis sensor saw (the tri-axis was on a different mount that responded
#     to whatever input was there). It is NOT a trigger/magnet artifact.
# (e) Single vs tri-axis: At-impact-window CFC180 ratio CH1/CH4 = 1.10-1.12 (not 1.55). CFC1000 ratio = 1.62-1.68.
#     The discrepancy is partly real frequency-content difference (different mounting/location) and partly the timing offset
#     (CH1 peak lagging CH4 by 250 µs at CFC1000) - all consistent with the two sensors being at different mechanical locations.

# Let me double-check (d) by examining CH4 signal in events 6,7,8 around 4 ms.
fig, axes = plt.subplots(3, 1, figsize=(10, 8))
for ax, n in zip(axes, [6, 7, 8]):
    df = events[n]
    t = df["Time (sec)"].values * 1000
    for ch in ["CH1 Acc (G's)","CH4 Acc (G's)"]:
        ax.plot(t, df[ch].values, label=ch.split()[0], lw=0.6)
    ax.set_xlim(0, 10); ax.legend(); ax.set_title(f"Event {n} (quiet) — CH1 and CH4 near 4.2 ms")
    ax.set_ylabel("G")
plt.tight_layout()
plt.savefig(f"{WD}/quiet_events.png", dpi=120); plt.close()
print("saved")

saved


In [35]:
# Final test: are the events 9, 10, 13 (CH1 peaks at exactly 932 G at 4.2 ms) also showing a 'fixed ceiling'?
# Note: 932, 932, 933 are too close to be coincidence given drop variability — but this isn't saturation 
# (the rounded peaks were smooth, not compressed). Could be:
#  - same sensor on same mount, dropped from same height repeatedly (the carriage rebound limiter etc.)
# Looking at the in-window CH1/CH4 ratio:
for n in [9, 10, 13]:
    r = ridge_df.loc[n]
    if abs(r["CH4_cfc1000_peakG"]) > 0.5:
        print(f"Event {n} CH1_cfc1000/CH4_cfc1000 in-window = {r['CH1_cfc1000_peakG']/r['CH4_cfc1000_peakG']:.2f}")
        print(f"  CH1 raw={r['CH1_raw_peakG']:.1f}, CH4 raw={r['CH4_raw_peakG']:.1f}")
        print(f"  CH4 sees almost nothing — different sensor positions, not co-located")
    else:
        print(f"Event {n} CH4 ~ 0; not a co-located test")

Event 9 CH1_cfc1000/CH4_cfc1000 in-window = 79.24
  CH1 raw=920.7, CH4 raw=15.3
  CH4 sees almost nothing — different sensor positions, not co-located
Event 10 CH1_cfc1000/CH4_cfc1000 in-window = 782.02
  CH1 raw=918.7, CH4 raw=1.5
  CH4 sees almost nothing — different sensor positions, not co-located
Event 13 CH1_cfc1000/CH4_cfc1000 in-window = 7.91
  CH1 raw=924.7, CH4 raw=155.1
  CH4 sees almost nothing — different sensor positions, not co-located


In [36]:
# I have enough material to deliver. Let me also briefly cross-reference SAE J211 for citation purposes via web search.
# And ISO 16063 for accelerometer calibration practice.

# Save a corrected peak summary table for upload as supporting artifact.
import pandas as pd
out = ridge_df.reset_index()
out["CH1_over_CH4_cfc180_in_window"] = out["CH1_cfc180_peakG"]/out["CH4_cfc180_peakG"]
out["CH1_over_CH4_cfc1000_in_window"] = out["CH1_cfc1000_peakG"]/out["CH4_cfc1000_peakG"]
keep = ["event","CH4_impact_time_ms",
        "CH1_raw_peakG","CH1_cfc1000_peakG","CH1_cfc180_peakG","CH1_cfc1000_peakT_ms","CH1_cfc180_peakT_ms",
        "CH4_raw_peakG","CH4_cfc1000_peakG","CH4_cfc180_peakG","CH4_cfc1000_peakT_ms","CH4_cfc180_peakT_ms",
        "CH1_over_CH4_cfc1000_in_window","CH1_over_CH4_cfc180_in_window"]
out[keep].round(3).to_csv(f"{WD}/peak_summary_impact_window.csv", index=False)
out[keep].round(2)

,event,CH4_impact_time_ms,CH1_raw_peakG,CH1_cfc1000_peakG,CH1_cfc180_peakG,CH1_cfc1000_peakT_ms,CH1_cfc180_peakT_ms,CH4_raw_peakG,CH4_cfc1000_peakG,CH4_cfc180_peakG,CH4_cfc1000_peakT_ms,CH4_cfc180_peakT_ms,CH1_over_CH4_cfc1000_in_window,CH1_over_CH4_cfc180_in_window
0,1,4.19,2585.65,1770.66,342.60,4.44,4.42,1275.46,1055.63,307.22,4.19,4.18,1.68,1.12
1,2,4.70,8714.75,8404.34,2634.25,3.74,3.71,-187.99,103.88,24.06,3.98,5.21,80.90,109.48
2,3,3.94,8782.97,8451.84,2634.40,3.74,3.72,227.67,153.24,25.78,3.98,3.90,55.15,102.21
3,4,4.18,2496.06,1728.63,339.59,4.43,4.41,1294.56,1068.83,309.46,4.19,4.18,1.62,1.10
4,5,3.95,8699.36,8357.82,2557.19,3.75,3.72,212.48,150.84,25.68,3.99,3.90,55.41,99.60
5,6,4.21,0.63,0.60,0.61,5.21,5.21,0.04,0.01,0.00,4.66,5.21,88.33,142.82
6,7,5.48,22.54,1.16,0.54,4.78,4.67,-0.17,-0.07,-0.01,6.16,4.48,-15.70,-76.39
7,8,3.86,-9.18,-1.46,-0.42,3.90,3.74,-0.52,-0.14,-0.08,3.90,3.98,10.56,5.50
8,9,4.38,920.70,881.42,258.77,4.20,4.14,15.26,11.12,1.84,4.43,4.34,79.24,140.88
9,10,4.39,918.74,879.79,256.86,4.20,4.14,1.54,1.13,0.19,4.43,4.34,782.02,1373.66


In [37]:
# Make a comparison figure of CH1 vs CH4 around impact for events 1 and 4, showing aligned-peak ratios.
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for col, n in enumerate([1, 4]):
    df = events[n]
    t = df["Time (sec)"].values * 1000
    s1 = df["CH1 Acc (G's)"].values - df["CH1 Acc (G's)"].values[:200].mean()
    s4 = df["CH4 Acc (G's)"].values - df["CH4 Acc (G's)"].values[:200].mean()
    s1_180 = j211(s1.copy(), 125000, 180)
    s1_1000 = j211(s1.copy(), 125000, 1000)
    s4_180 = j211(s4.copy(), 125000, 180)
    s4_1000 = j211(s4.copy(), 125000, 1000)

    ax = axes[0, col]
    ax.plot(t, s1, 'k-', lw=0.4, alpha=0.5, label='CH1 raw')
    ax.plot(t, s1_1000, 'b-', lw=1.0, label='CH1 CFC1000')
    ax.plot(t, s1_180, 'r-', lw=1.2, label='CH1 CFC180')
    ax.axvline(4.18, color='g', ls=':', alpha=0.6, label='CH4 impact')
    ax.set_xlim(0, 30); ax.set_xlabel('ms'); ax.set_ylabel('CH1 (G)')
    ax.set_title(f"Event {n}: CH1 — CFC180 peak is at {t[np.argmax(s1_180)]:.1f} ms (LF post-impact!)")
    ax.legend(fontsize=8)

    ax = axes[1, col]
    ax.plot(t, s4, 'k-', lw=0.4, alpha=0.5, label='CH4 raw')
    ax.plot(t, s4_1000, 'b-', lw=1.0, label='CH4 CFC1000')
    ax.plot(t, s4_180, 'r-', lw=1.2, label='CH4 CFC180')
    ax.axvline(4.18, color='g', ls=':', alpha=0.6, label='impact')
    ax.set_xlim(0, 30); ax.set_xlabel('ms'); ax.set_ylabel('CH4 (G)')
    ax.set_title(f"Event {n}: CH4 — CFC180 peak at impact 4.18 ms")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{WD}/reviewer_ch1_ch4_alignment.png", dpi=130)
plt.close()
print("Saved reviewer figure")

Saved reviewer figure


In [38]:
# Upload artifacts so the user can pull them
from e14c.data_storage import upload
import os
os.environ['PROJECT_TAG'] = os.environ.get('PROJECT_TAG') or 'accelerometer-tuning-review'

uri_csv = upload(f"{WD}/peak_summary_impact_window.csv",
                 description="Reviewer's corrected peak table: per-event raw, CFC1000, CFC180 peaks restricted to ±1 ms of CH4 impact, with peak times.",
                 name="peak_summary_impact_window.csv")
uri_fig = upload(f"{WD}/reviewer_ch1_ch4_alignment.png",
                 description="Events 1 & 4: CH1 vs CH4 raw / CFC1000 / CFC180 — shows CH1 CFC180 peak occurs at 15.8 ms (LF post-impact), not at the 4.18 ms impact.",
                 name="reviewer_ch1_ch4_alignment.png")
print(uri_csv); print(uri_fig)

Uploaded /workspace/015f36e1-0a1c-4aed-a9a3-1d1924983c4a/peak_summary_impact_window.csv to data storage as entry data_entry:peak-summary-impact-window-csv-z4br under the name 'peak_summary_impact_window.csv'.


Uploaded /workspace/015f36e1-0a1c-4aed-a9a3-1d1924983c4a/reviewer_ch1_ch4_alignment.png to data storage as entry data_entry:reviewer-ch1-ch4-alignment-png-l5fg under the name 'reviewer_ch1_ch4_alignment.png'.
data_entry:peak-summary-impact-window-csv-z4br
data_entry:reviewer-ch1-ch4-alignment-png-l5fg
